# T06 - 城投区域分析

## 项目概述

城投区域分析项目专注于地方政府融资平台（城投债）的区域性信用风险分析和投资策略研究。

### 主要功能
1. **区域风险评级** - 建立区域信用风险评级体系
2. **利差分析** - 分析不同区域城投债的利差走势
3. **投资策略** - 提供区域配置建议和风险预警
4. **政策跟踪** - 跟踪地方债务政策变化

### 数据源
- Wind数据库
- 同花顺iFinD
- 财政部官网
- 国家统计局

---
## 1. 环境配置

In [ ]:
# 导入标准库
import os
import sys
import datetime
import warnings
warnings.filterwarnings('ignore')

# 数据处理
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import sqlalchemy.pool

# 可视化
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

# 导入配置
from config import (
    DB_CONFIG, 
    RATING_CONFIG, 
    VIZ_CONFIG,
    DATA_DIR,
    OUTPUT_DIR,
    get_db_connection_string,
    ensure_directories
)

# 确保目录存在
ensure_directories()

print("环境配置完成！")
print(f"数据目录: {DATA_DIR}")
print(f"输出目录: {OUTPUT_DIR}")

In [ ]:
# 设置中文字体
def setup_chinese_font():
    """设置matplotlib中文字体"""
    plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
    plt.rcParams['axes.unicode_minus'] = False
    print("中文字体设置完成")

setup_chinese_font()

---
## 2. 数据获取

### 2.1 数据库连接

In [ ]:
# 创建数据库连接
def create_db_connection():
    """创建数据库连接引擎"""
    try:
        connection_string = get_db_connection_string()
        engine = create_engine(
            connection_string,
            poolclass=sqlalchemy.pool.NullPool,
            pool_recycle=3600
        )
        print("数据库连接创建成功")
        return engine
    except Exception as e:
        print(f"数据库连接失败: {e}")
        return None

db_engine = create_db_connection()

### 2.2 读取城投平台基础数据

In [ ]:
def load_platform_data(engine, limit=None):
    """
    从数据库读取城投平台基础数据
    
    Parameters:
    -----------
    engine : SQLAlchemy Engine
        数据库连接引擎
    limit : int, optional
        限制返回的记录数
    
    Returns:
    --------
    pd.DataFrame
        城投平台数据
    """
    sql = """
    SELECT 
        trade_code,
        platform_name,
        region_province,
        region_city,
        region_district,
        admin_level,
        registered_capital,
        bond_type,
        issue_scale,
        remaining_years,
        bond_rating,
        platform_rating,
        region_rating
    FROM bond.basicinfo_lgf
    WHERE 1=1
    """
    
    if limit:
        sql += f" LIMIT {limit}"
    
    try:
        with engine.begin() as conn:
            df = pd.read_sql(sql, conn)
        print(f"成功读取 {len(df)} 条城投平台数据")
        return df
    except Exception as e:
        print(f"读取城投平台数据失败: {e}")
        return pd.DataFrame()

# 示例：读取数据（如果数据库可用）
# df_platform = load_platform_data(db_engine)
# df_platform.head()

### 2.3 读取区域经济数据

In [ ]:
def load_region_economic_data(engine, year=None):
    """
    从数据库读取区域经济数据
    
    Parameters:
    -----------
    engine : SQLAlchemy Engine
        数据库连接引擎
    year : int, optional
        筛选年份
    
    Returns:
    --------
    pd.DataFrame
        区域经济数据
    """
    sql = """
    SELECT 
        region_code,
        region_name,
        admin_level,
        year,
        quarter,
        gdp,
        gdp_growth,
        gdp_per_capita,
        general_budget_revenue,
        govt_fund_revenue,
        revenue_growth,
        fiscal_self_support_ratio,
        local_debt_balance,
        debt_ratio,
        liability_ratio,
        deposit_balance,
        loan_balance
    FROM bond.region_economic_data
    WHERE 1=1
    """
    
    if year:
        sql += f" AND year = {year}"
    
    try:
        with engine.begin() as conn:
            df = pd.read_sql(sql, conn)
        print(f"成功读取 {len(df)} 条区域经济数据")
        return df
    except Exception as e:
        print(f"读取区域经济数据失败: {e}")
        return pd.DataFrame()

# 示例：读取数据（如果数据库可用）
# df_region = load_region_economic_data(db_engine, year=2024)
# df_region.head()

### 2.4 读取本地Excel数据

In [ ]:
def load_local_excel_data(file_name):
    """
    读取本地Excel数据文件
    
    Parameters:
    -----------
    file_name : str
        Excel文件名
    
    Returns:
    --------
    pd.DataFrame
        Excel数据
    """
    file_path = os.path.join(DATA_DIR, file_name)
    
    if not os.path.exists(file_path):
        print(f"文件不存在: {file_path}")
        return pd.DataFrame()
    
    try:
        df = pd.read_excel(file_path)
        print(f"成功读取文件: {file_name}")
        print(f"数据形状: {df.shape}")
        return df
    except Exception as e:
        print(f"读取Excel文件失败: {e}")
        return pd.DataFrame()

# 读取西安城投数据
df_xian = load_local_excel_data('西安城投数据.xlsx')
if not df_xian.empty:
    display(df_xian.head())

---
## 3. 数据处理

### 3.1 数据清洗

In [ ]:
def clean_data(df, critical_columns=None):
    """
    数据清洗函数
    
    Parameters:
    -----------
    df : pd.DataFrame
        原始数据
    critical_columns : list, optional
        关键字段列表，这些字段不能为空
    
    Returns:
    --------
    pd.DataFrame
        清洗后的数据
    """
    df_clean = df.copy()
    original_count = len(df_clean)
    
    # 1. 删除关键字段缺失的记录
    if critical_columns:
        df_clean = df_clean.dropna(subset=critical_columns)
        print(f"删除关键字段缺失记录: {original_count - len(df_clean)} 条")
    
    # 2. 删除完全重复的记录
    duplicates = df_clean.duplicated().sum()
    if duplicates > 0:
        df_clean = df_clean.drop_duplicates()
        print(f"删除重复记录: {duplicates} 条")
    
    # 3. 数值型缺失值填充（使用中位数）
    numeric_columns = df_clean.select_dtypes(include=[np.number]).columns
    for col in numeric_columns:
        if df_clean[col].isna().any():
            median_val = df_clean[col].median()
            df_clean[col] = df_clean[col].fillna(median_val)
    
    print(f"数据清洗完成: {original_count} -> {len(df_clean)} 条记录")
    return df_clean

# 示例：清洗数据
# df_clean = clean_data(df_xian, critical_columns=['platform_name', 'region_province'])

### 3.2 指标计算

In [ ]:
def calculate_fiscal_indicators(df):
    """
    计算财政实力指标
    
    Parameters:
    -----------
    df : pd.DataFrame
        区域经济数据
    
    Returns:
    --------
    pd.DataFrame
        添加财政指标后的数据
    """
    df_result = df.copy()
    
    # 财政自给率 = 一般公共预算收入 / 一般公共预算支出
    if 'general_budget_revenue' in df.columns and 'general_budget_expenditure' in df.columns:
        df_result['fiscal_self_sufficiency'] = (
            df_result['general_budget_revenue'] / df_result['general_budget_expenditure'] * 100
        )
    
    # 土地财政依赖度 = 政府性基金收入 / 综合财力
    if 'govt_fund_revenue' in df.columns and 'general_budget_revenue' in df.columns:
        df_result['land_fiscal_dependency'] = (
            df_result['govt_fund_revenue'] / 
            (df_result['general_budget_revenue'] + df_result['govt_fund_revenue']) * 100
        )
    
    return df_result


def calculate_debt_indicators(df):
    """
    计算债务负担指标
    
    Parameters:
    -----------
    df : pd.DataFrame
        区域经济数据
    
    Returns:
    --------
    pd.DataFrame
        添加债务指标后的数据
    """
    df_result = df.copy()
    
    # 债务率 = 地方政府债务余额 / GDP
    if 'local_debt_balance' in df.columns and 'gdp' in df.columns:
        df_result['debt_to_gdp'] = df_result['local_debt_balance'] / df_result['gdp'] * 100
    
    # 负债率 = 地方政府债务余额 / 综合财力
    if 'local_debt_balance' in df.columns and 'general_budget_revenue' in df.columns:
        df_result['debt_to_revenue'] = (
            df_result['local_debt_balance'] / df_result['general_budget_revenue'] * 100
        )
    
    return df_result


def calculate_all_indicators(df):
    """
    计算所有指标
    
    Parameters:
    -----------
    df : pd.DataFrame
        区域经济数据
    
    Returns:
    --------
    pd.DataFrame
        添加所有指标后的数据
    """
    df_result = df.copy()
    df_result = calculate_fiscal_indicators(df_result)
    df_result = calculate_debt_indicators(df_result)
    print("指标计算完成")
    return df_result

### 3.3 利差计算

In [ ]:
def calculate_credit_spread(bond_yield, treasury_yield):
    """
    计算信用利差
    
    Parameters:
    -----------
    bond_yield : float or pd.Series
        城投债收益率
    treasury_yield : float or pd.Series
        同期限国债收益率
    
    Returns:
    --------
    float or pd.Series
        信用利差（基点）
    """
    return (bond_yield - treasury_yield) * 100  # 转换为基点


def calculate_spread_percentile(spread_series):
    """
    计算利差分位数
    
    Parameters:
    -----------
    spread_series : pd.Series
        利差时间序列
    
    Returns:
    --------
    float
        当前利差的历史分位数（0-1）
    """
    if len(spread_series) < 2:
        return np.nan
    
    current_spread = spread_series.iloc[-1]
    historical_spreads = spread_series.iloc[:-1]
    percentile = (historical_spreads < current_spread).sum() / len(historical_spreads)
    return percentile

---
## 4. 核心逻辑

### 4.1 区域风险评级

In [ ]:
def calculate_region_rating(indicators):
    """
    计算区域风险评级
    
    Parameters:
    -----------
    indicators : dict
        包含各项指标得分的字典
        - 财政实力: 0-100
        - 债务负担: 0-100
        - 经济活力: 0-100
        - 金融支持: 0-100
        - 政策环境: 0-100
    
    Returns:
    --------
    tuple
        (评级, 综合评分)
    """
    weights = RATING_CONFIG['区域评级']['weights']
    thresholds = RATING_CONFIG['区域评级']['thresholds']
    
    # 计算综合评分
    score = sum(
        indicators.get(key, 0) * weight 
        for key, weight in weights.items()
    )
    
    # 确定评级
    if score >= thresholds['AAA']:
        rating = 'AAA'
    elif score >= thresholds['AA+']:
        rating = 'AA+'
    elif score >= thresholds['AA']:
        rating = 'AA'
    elif score >= thresholds['AA-']:
        rating = 'AA-'
    else:
        rating = 'A'
    
    return rating, round(score, 2)


# 示例：计算区域评级
example_indicators = {
    '财政实力': 85,
    '债务负担': 75,
    '经济活力': 80,
    '金融支持': 70,
    '政策环境': 75
}
rating, score = calculate_region_rating(example_indicators)
print(f"区域评级: {rating}, 综合评分: {score}")

### 4.2 平台风险评级

In [ ]:
def calculate_platform_rating(indicators):
    """
    计算平台风险评级
    
    Parameters:
    -----------
    indicators : dict
        包含各项指标得分的字典
        - 股东背景: 0-100
        - 区域风险: 0-100
        - 财务状况: 0-100
        - 市场表现: 0-100
        - 增信情况: 0-100
    
    Returns:
    --------
    tuple
        (评级, 综合评分)
    """
    weights = RATING_CONFIG['平台评级']['weights']
    thresholds = RATING_CONFIG['平台评级']['thresholds']
    
    # 计算综合评分
    score = sum(
        indicators.get(key, 0) * weight 
        for key, weight in weights.items()
    )
    
    # 确定评级
    if score >= thresholds['AAA']:
        rating = 'AAA'
    elif score >= thresholds['AA+']:
        rating = 'AA+'
    elif score >= thresholds['AA']:
        rating = 'AA'
    elif score >= thresholds['AA-']:
        rating = 'AA-'
    else:
        rating = 'A'
    
    return rating, round(score, 2)


# 示例：计算平台评级
example_platform_indicators = {
    '股东背景': 80,
    '区域风险': 75,
    '财务状况': 70,
    '市场表现': 65,
    '增信情况': 80
}
platform_rating, platform_score = calculate_platform_rating(example_platform_indicators)
print(f"平台评级: {platform_rating}, 综合评分: {platform_score}")

### 4.3 投资策略分析

In [ ]:
def get_investment_strategy(rating, spread_percentile):
    """
    根据评级和利差分位数获取投资策略建议
    
    Parameters:
    -----------
    rating : str
        区域或平台评级
    spread_percentile : float
        利差分位数（0-1）
    
    Returns:
    --------
    dict
        投资策略建议
    """
    strategy = {
        'rating': rating,
        'spread_percentile': spread_percentile,
        'strategy_type': '',
        'position_advice': '',
        'duration_advice': '',
        'risk_control': ''
    }
    
    # 根据评级确定基础策略
    if rating in ['AAA', 'AA+']:
        strategy['strategy_type'] = '核心区域策略'
        strategy['position_advice'] = '重仓配置'
        strategy['duration_advice'] = '久期3-5年'
        strategy['risk_control'] = '关注财政实力变化'
    elif rating == 'AA':
        strategy['strategy_type'] = '成长区域策略'
        strategy['position_advice'] = '适度配置'
        strategy['duration_advice'] = '久期2-4年'
        strategy['risk_control'] = '关注经济增速'
    else:
        strategy['strategy_type'] = '机会区域策略'
        strategy['position_advice'] = '轻仓配置'
        strategy['duration_advice'] = '久期1-3年'
        strategy['risk_control'] = '严格控制风险敞口'
    
    # 根据利差分位数调整
    if spread_percentile > 0.8:
        strategy['timing_signal'] = '超配机会（利差分位数>80%）'
    elif spread_percentile < 0.2:
        strategy['timing_signal'] = '减仓信号（利差分位数<20%）'
    else:
        strategy['timing_signal'] = '中性（利差分位数正常）'
    
    return strategy


# 示例：获取投资策略
strategy = get_investment_strategy('AA', 0.75)
print("投资策略建议:")
for key, value in strategy.items():
    print(f"  {key}: {value}")

---
## 5. 执行与测试

### 5.1 主流程执行

In [ ]:
def main_analysis_pipeline(data_source='local'):
    """
    主分析流程
    
    Parameters:
    -----------
    data_source : str
        数据源类型 ('local' 或 'database')
    
    Returns:
    --------
    dict
        分析结果
    """
    results = {
        'status': 'success',
        'data_count': 0,
        'ratings': [],
        'strategies': []
    }
    
    print("="*50)
    print("城投区域分析 - 主流程执行")
    print("="*50)
    
    # 1. 数据获取
    print("\n[1] 数据获取...")
    if data_source == 'local':
        df = load_local_excel_data('西安城投数据.xlsx')
    else:
        df = load_platform_data(db_engine)
    
    if df.empty:
        print("警告: 未获取到数据")
        results['status'] = 'no_data'
        return results
    
    results['data_count'] = len(df)
    
    # 2. 数据清洗
    print("\n[2] 数据清洗...")
    df_clean = clean_data(df)
    
    # 3. 指标计算
    print("\n[3] 指标计算...")
    df_with_indicators = calculate_all_indicators(df_clean)
    
    # 4. 示例评级计算
    print("\n[4] 示例评级计算...")
    sample_indicators = {
        '财政实力': 80,
        '债务负担': 70,
        '经济活力': 75,
        '金融支持': 65,
        '政策环境': 70
    }
    rating, score = calculate_region_rating(sample_indicators)
    results['ratings'].append({'type': 'sample', 'rating': rating, 'score': score})
    print(f"样本评级: {rating}, 评分: {score}")
    
    # 5. 投资策略
    print("\n[5] 投资策略分析...")
    strategy = get_investment_strategy(rating, 0.65)
    results['strategies'].append(strategy)
    
    print("\n" + "="*50)
    print("分析完成!")
    print("="*50)
    
    return results

# 执行主流程
results = main_analysis_pipeline(data_source='local')
print(f"\n执行状态: {results['status']}")
print(f"数据条数: {results['data_count']}")

### 5.2 结果验证

In [ ]:
def validate_results(results):
    """
    验证分析结果
    
    Parameters:
    -----------
    results : dict
        分析结果
    
    Returns:
    --------
    bool
        验证是否通过
    """
    print("\n结果验证:")
    print("-" * 30)
    
    # 检查状态
    if results['status'] != 'success':
        print(f"[X] 执行状态异常: {results['status']}")
        return False
    print("[V] 执行状态正常")
    
    # 检查数据量
    if results['data_count'] == 0:
        print("[X] 数据量为0")
        return False
    print(f"[V] 数据量: {results['data_count']}")
    
    # 检查评级
    if not results['ratings']:
        print("[X] 无评级结果")
        return False
    print(f"[V] 评级结果: {results['ratings']}")
    
    print("-" * 30)
    print("验证通过!")
    return True

# 验证结果
is_valid = validate_results(results)

### 5.3 可视化输出

In [ ]:
def plot_rating_distribution(ratings_data):
    """
    绘制评级分布图
    
    Parameters:
    -----------
    ratings_data : list
        评级数据列表
    """
    # 示例数据
    rating_counts = {'AAA': 5, 'AA+': 15, 'AA': 30, 'AA-': 20, 'A': 10}
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    colors = [VIZ_CONFIG['color_palette'][r] for r in rating_counts.keys()]
    bars = ax.bar(rating_counts.keys(), rating_counts.values(), color=colors)
    
    ax.set_xlabel('评级', fontsize=12)
    ax.set_ylabel('数量', fontsize=12)
    ax.set_title('城投区域评级分布', fontsize=14)
    
    # 添加数值标签
    for bar, count in zip(bars, rating_counts.values()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                str(count), ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    
    # 保存图片
    output_path = os.path.join(OUTPUT_DIR, 'rating_distribution.png')
    plt.savefig(output_path, dpi=VIZ_CONFIG['dpi'], bbox_inches='tight')
    print(f"图表已保存: {output_path}")
    
    plt.show()

# 绘制评级分布图
plot_rating_distribution(results.get('ratings', []))

In [ ]:
def plot_spread_trend():
    """
    绘制利差走势图（示例）
    """
    # 生成示例数据
    dates = pd.date_range(start='2024-01-01', periods=12, freq='M')
    aaa_spread = [50 + np.random.randn()*5 for _ in range(12)]
    aa_spread = [80 + np.random.randn()*8 for _ in range(12)]
    aa_minus_spread = [120 + np.random.randn()*10 for _ in range(12)]
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    ax.plot(dates, aaa_spread, 'o-', label='AAA', color=VIZ_CONFIG['color_palette']['AAA'], linewidth=2)
    ax.plot(dates, aa_spread, 's-', label='AA', color=VIZ_CONFIG['color_palette']['AA'], linewidth=2)
    ax.plot(dates, aa_minus_spread, '^-', label='AA-', color=VIZ_CONFIG['color_palette']['AA-'], linewidth=2)
    
    ax.set_xlabel('日期', fontsize=12)
    ax.set_ylabel('信用利差 (bp)', fontsize=12)
    ax.set_title('城投债信用利差走势', fontsize=14)
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    # 保存图片
    output_path = os.path.join(OUTPUT_DIR, 'spread_trend.png')
    plt.savefig(output_path, dpi=VIZ_CONFIG['dpi'], bbox_inches='tight')
    print(f"图表已保存: {output_path}")
    
    plt.show()

# 绘制利差走势图
plot_spread_trend()

---
## 6. 工具函数

In [ ]:
def save_to_excel(df, filename, sheet_name='Sheet1'):
    """
    保存数据到Excel文件
    
    Parameters:
    -----------
    df : pd.DataFrame
        要保存的数据
    filename : str
        文件名
    sheet_name : str
        工作表名
    """
    output_path = os.path.join(OUTPUT_DIR, filename)
    df.to_excel(output_path, sheet_name=sheet_name, index=False)
    print(f"数据已保存: {output_path}")


def normalize_score(value, min_val, max_val, reverse=False):
    """
    将数值标准化为0-100的评分
    
    Parameters:
    -----------
    value : float
        原始值
    min_val : float
        最小值
    max_val : float
        最大值
    reverse : bool
        是否反向（值越大评分越低）
    
    Returns:
    --------
    float
        标准化评分 (0-100)
    """
    if max_val == min_val:
        return 50
    
    score = (value - min_val) / (max_val - min_val) * 100
    
    if reverse:
        score = 100 - score
    
    return max(0, min(100, score))


def calculate_weighted_score(indicators, weights):
    """
    计算加权评分
    
    Parameters:
    -----------
    indicators : dict
        指标字典 {指标名: 值}
    weights : dict
        权重字典 {指标名: 权重}
    
    Returns:
    --------
    float
        加权评分
    """
    total_weight = sum(weights.values())
    weighted_sum = sum(
        indicators.get(key, 0) * weight 
        for key, weight in weights.items()
    )
    return weighted_sum / total_weight if total_weight > 0 else 0


def generate_summary_report(results):
    """
    生成摘要报告
    
    Parameters:
    -----------
    results : dict
        分析结果
    
    Returns:
    --------
    str
        摘要报告文本
    """
    report = []
    report.append("="*50)
    report.append("城投区域分析摘要报告")
    report.append("="*50)
    report.append(f"分析日期: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    report.append(f"执行状态: {results['status']}")
    report.append(f"数据条数: {results['data_count']}")
    report.append("")
    
    if results['ratings']:
        report.append("评级结果:")
        for r in results['ratings']:
            report.append(f"  - {r['type']}: {r['rating']} (评分: {r['score']})")
    
    if results['strategies']:
        report.append("")
        report.append("投资策略:")
        for s in results['strategies']:
            report.append(f"  - 类型: {s['strategy_type']}")
            report.append(f"    仓位建议: {s['position_advice']}")
            report.append(f"    久期建议: {s['duration_advice']}")
    
    report.append("")
    report.append("="*50)
    
    return "\n".join(report)


# 生成摘要报告
report = generate_summary_report(results)
print(report)

# 保存报告
report_path = os.path.join(OUTPUT_DIR, 'summary_report.txt')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report)
print(f"\n报告已保存: {report_path}")

---
## 总结

本Notebook完成了城投区域分析的以下功能：

1. **环境配置** - 导入依赖库和配置参数
2. **数据获取** - 支持数据库和本地文件两种数据源
3. **数据处理** - 数据清洗和指标计算
4. **核心逻辑** - 区域风险评级、平台评级和投资策略分析
5. **执行与测试** - 主流程执行和结果验证
6. **工具函数** - 可复用的辅助函数

### 后续扩展方向

- 接入Wind API获取实时数据
- 添加更多可视化图表
- 实现自动化报告生成
- 集成同花顺iFinD数据源